# Quantum Teleportation Lab
# Part 2a: Alice sends a message
## SIGGRAPH 2026, Los Angeles, CA, July 2026
## Andrew Glassner 

www.glassner.com

Copyright 2026 by Andrew Glassner. Licensed under CC BY-NC 4.0

This notebook and its associated notes can be found on https://github.com/blueberrymusic/Quantum-Teleportation

---

<div style="background-color: #f0fff0; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #2E8B57; margin-top: 0;">Alice's Infrastructure</h1></center>
<center><h3>Utility code</h3></center>
<ul>
<li><strong>Import libraries</strong></li>
<li><strong>make_Alices_QC - build Alice's circuit</strong></li>
<li><strong>run_Alices_QC - run Alice's circuit</strong></li>
<li><strong>execute_Alices_circuit - combined build & run</li>
</ul
</div>


### Import all the packages we'll use

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister 
from qiskit.quantum_info import Statevector

import numpy as np
import matplotlib.pyplot as plt

# This is "magic" that lets Qiskit draw into notebook cells
%matplotlib inline

### Build Alice's circuit

In [ ]:
def make_Alices_QC(alpha, create_gates):
  
    # Though alpha and beta are real numbers in this demo, it's a good habit 
    # to write amplitudes as complex numbers (with a 0 imaginary component).
    sigma_alpha = alpha + 0j
    sigma_beta = np.sqrt(1-(sigma_alpha**2)) # Because |alpha|^2 + |beta|^2 = 1
    sigma_state = np.array([sigma_alpha, sigma_beta])
        
    qs = QuantumRegister(1, 's')[0]   # Source qubit to teleport
    qa = QuantumRegister(1, 'a')[0]   # Alice's qubit
    qb = QuantumRegister(1, 'b')[0]   # Bob's qubit
   
    # Create the circuit. We don't need classical bits to collect
    # measurements because this simulator doesnt support measurement
    Alices_QC = QuantumCircuit([qs, qa, qb])
   
    # Initialize the qubits. qs gets the sigma_state we computed above,
    # which is the state we want to teleport. Following convention, the
    # others default to |0>.
    Alices_QC.initialize(sigma_state, qs)

    # Insert the gates of Alice's circuit
    create_gates(Alices_QC, qs, qa, qb)

    # Now we've built the quantum circuit, up to measurement, so return
    # it. We manually implement measurement in run_Alices_step()
    return Alices_QC

### Run Alice's circuit, "measure" outputs and return them, also Bob's qubit

In [ ]:
def run_Alices_QC(Alices_QC):
    
    # Evaluate Alice's circuit.
    # We're using the Statevector simulator because we want to be able
    # to obtain the explicit amplitudes on Bob's qubit. Unfortunately,
    # that simulator doesn't support measurement. So in this function,
    # we simulate quantum measurement by doing what Nature does. We start
    # by computing the probabilities of each measurement, and then we
    # extract a random measurement from the superposition based on
    # those probabilities (albeit with a pseudo-random number generator,
    # not the actual randomness we see in the physical world).
    
    # The Statevector simulator runs the circuit from the start, and
    # returns the amplitudes of the 3-qubit system of qubits at the 
    # end of the circuit
    bottom_up_amplitudes = Statevector.from_instruction(Alices_QC)
    
    # Qiskit reports amplitudes bottom-up, but I prefer top-down, so 
    # let's swap them around by reversing the bitstrings, e.g. state
    # 3 (011) becomes state 6 (110)
    amplitudes = bottom_up_amplitudes.data[[0, 4, 2, 6, 1, 5, 3, 7]]

    # Convert the amplitudes to probabilities by squaring them
    probabilities = np.abs(amplitudes)**2  

    # Find the probabilities that measuring qs and qa will give
    # us 00, 01, 10, or 11. For example, the probablility of
    # measuring 00 is the sum of the probabilities of measuring
    # 000 and 001, the probability of 01 is the sum of 010 and 011,
    # and so on.
    pair_probabilities = [
        probabilities[0] + probabilities[1], # 00x: 000 and 001
        probabilities[2] + probabilities[3], # 01x: 010 and 011
        probabilities[4] + probabilities[5], # 10x: 100 and 101
        probabilities[6] + probabilities[7]  # 11x: 110 and 111
    ]
    
    # "Measure" qubits 0 and 1 by sampling the probabilties. That is, given
    # the probabilities we just computed, select one of the four states
    # 00 through 11 based on their probabilities. numpy has a built-in
    # function random.choice() that does just this for us.
    measured_outcome = np.random.choice(4, p=pair_probabilities)
    
    # now that we have a "measured" outcome, reconstruct the two
    # bitstrings corresponding to that measurement. For example, if
    # the measured outcome was 2, we want the bitstring 10
    binary_measured = format(measured_outcome, '02b')

    bit_ms = int(binary_measured[0]) # the signal qubit
    bit_ma = int(binary_measured[1]) # Alice's qubit

    # Now compute the state on Bob's qubit. Suppose we measured 10. Then
    # the amplitude 100 is the amplitude of Bob's |0> and 101 is the amplitude
    # of Bob's |1>. Make that state as an array.
    state0_index = int(binary_measured + '0', 2)
    state1_index = int(binary_measured + '1', 2)
    qb_state = np.array([amplitudes[state0_index], amplitudes[state1_index]], dtype=complex)
        
    # Normalize Bob's state by setting the sum its squared values to 1
    norm = np.sqrt(np.abs(qb_state[0])**2 + np.abs(qb_state[1])**2)
    if norm > 0:
        qb_state = qb_state / norm
   
    return bit_ms, bit_ma, qb_state

### Perform Alice's phase

In [ ]:
def execute_Alices_circuit(alpha, create_gates):
    
    # Build Alice's circuit
    Alices_QC = make_Alices_QC(alpha, create_gates)
    
    # Execute Alice's circuit
    bit_ms, bit_ma, qb_state = run_Alices_QC(Alices_QC)
    
    filename = f'qb_state_{np.random.randint(10000, 1000000)}.txt'
    np.savetxt(filename, qb_state)
    
    return Alices_QC, bit_ms, bit_ma, qb_state, filename 

<div style="background-color: #fff0f8; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #882E57; margin-top: 0;">Stop entering cells here</h1></center>
</div>

---

<div style="background-color: #f0f8ff; padding: 15px; border-left: 5px solid #4CAF50; margin: 10px 0;">
<center><h1 style="color: #2E8B57; margin-top: 0;">Alice's Step</h1></center>
<center><h3>Send the secret message</h3></center>
<ul>
<li><strong>Define Alice's circuit</strong></li>
<li><strong>Declare the message</strong></li>
<li><strong>Send the message</strong></li>
</ul>
</div>

### Alice Step 1: Provide the quantum gates that make up Alice's circuit, in order left to right

In [ ]:
def create_Alices_gates(Alices_QC, qs, qa, qb):
    # Your code goes here

### Alice Step 2: Enter alpha: a float in [0,1]
Use 8 digits or fewer to the right of the decimal point.

In [ ]:
# Alice's secret message
alpha = # Your number between 0 and 1 goes here

### Alice Step 3: Build Alice's circuit, run it, get results

---

In [ ]:
Alices_QC, bit_ms, bit_ma, qb_state, filename = execute_Alices_circuit(alpha, create_Alices_gates)
    
# Draw Alice's quantum circuit, and print out the measured bits and filename for Bob
Alices_QC.draw('mpl')
print('filename =',filename,'\nbit_ms =',bit_ms,'\nbit_ma =',bit_ma)

---

# End of Part 2a: Quantum Teleportation